# 用純 Python 從零實作簡單大語言模型（LLM）

本筆記本展示一個最小可運行的 Transformer 語言模型，**不依賴 NumPy 或 PyTorch**，全部用純 Python 實作。

架構包含：
1. Token Embedding
2. Positional Encoding
3. Self-Attention
4. Feed Forward Network
5. 訓練迴圈（Cross-Entropy Loss + 梯度下降）
6. 文字生成

## 1. 超參數與詞彙表設定

In [1]:
import math
import random

# 字元級詞彙表
VOCAB = list("abcdefghijklmnopqrstuvwxyz .,!?\n")
V = len(VOCAB)  # 詞彙表大小
D = 32  # d_model：隱藏層維度
D_FF = 64  # Feed Forward 中間層維度
H = 2  # 注意力頭數
SEQ = 16  # 最大序列長度
LR = 0.005  # 學習率

# 字元 ↔ ID 對照表
char2id = {c: i for i, c in enumerate(VOCAB)}
id2char = {i: c for c, i in char2id.items()}

print(f"詞彙表大小: {V}")
print(f"字元範例: {VOCAB[:10]}")

詞彙表大小: 32
字元範例: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']


## 2. 基礎矩陣運算工具

用純 Python 實作矩陣乘法、softmax、ReLU 等基本運算。

In [2]:
def zeros(r, c):
    """建立全零矩陣"""
    return [[0.0] * c for _ in range(r)]


def randn(r, c, scale=0.1):
    """建立隨機初始化矩陣（常態分佈）"""
    return [[random.gauss(0, scale) for _ in range(c)] for _ in range(r)]


def matmul(A, B):
    """矩陣乘法 A @ B"""
    Ra, Ca = len(A), len(A[0])
    Rb, Cb = len(B), len(B[0])
    assert Ca == Rb, f"維度不符: {Ca} != {Rb}"
    C = zeros(Ra, Cb)
    for i in range(Ra):
        for k in range(Ca):
            if A[i][k] == 0.0:
                continue
            for j in range(Cb):
                C[i][j] += A[i][k] * B[k][j]
    return C


def transpose(A):
    """矩陣轉置"""
    return [[A[r][c] for r in range(len(A))] for c in range(len(A[0]))]


def add_vec(a, b):
    """向量相加"""
    return [x + y for x, y in zip(a, b)]


def scale_vec(v, s):
    """向量純量乘法"""
    return [x * s for x in v]


def softmax(v):
    """數值穩定的 softmax"""
    m = max(v)
    e = [math.exp(x - m) for x in v]
    s = sum(e)
    return [x / s for x in e]


def relu(v):
    """ReLU 激活函數"""
    return [max(0.0, x) for x in v]


def layer_norm(v, eps=1e-6):
    """Layer Normalization"""
    mean = sum(v) / len(v)
    var = sum((x - mean) ** 2 for x in v) / len(v)
    return [(x - mean) / math.sqrt(var + eps) for x in v]


print("工具函數定義完成 ✓")
# 快速測試
A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
print("matmul test:", matmul(A, B))  # [[19,22],[43,50]]
print("softmax test:", [f"{x:.3f}" for x in softmax([1.0, 2.0, 3.0])])

工具函數定義完成 ✓
matmul test: [[19.0, 22.0], [43.0, 50.0]]
softmax test: ['0.090', '0.245', '0.665']


## 3. Token Embedding 層

將每個 Token ID 映射到一個 D 維向量。

In [3]:
class Embedding:
    """
    查找表：Token ID → D 維向量
    形狀：W[vocab_size][d_model]
    """

    def __init__(self, vocab_size, d_model):
        self.W = randn(vocab_size, d_model, scale=0.02)

    def forward(self, ids):
        """ids: list of int → list of D 維向量"""
        return [self.W[i][:] for i in ids]

    def params(self):
        return [self.W]


# 示範
emb = Embedding(V, D)
test_ids = [char2id["h"], char2id["i"]]
vecs = emb.forward(test_ids)
print(f"輸入 IDs: {test_ids}  →  輸出向量形狀: {len(vecs)} x {len(vecs[0])}")
print(f"第一個向量（前 5 維）: {[f'{x:.4f}' for x in vecs[0][:5]]}")

輸入 IDs: [7, 8]  →  輸出向量形狀: 2 x 32
第一個向量（前 5 維）: ['0.0224', '-0.0234', '0.0512', '-0.0197', '0.0168']


## 4. Positional Encoding

用 sin/cos 函數為每個位置加入位置資訊，讓模型知道詞的順序。

In [4]:
def positional_encoding(seq_len, d_model):
    """
    標準 Transformer Positional Encoding
    PE[pos][2i]   = sin(pos / 10000^(2i/d_model))
    PE[pos][2i+1] = cos(pos / 10000^(2i/d_model))
    """
    pe = zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (i / d_model))
            pe[pos][i] = math.sin(angle)
            if i + 1 < d_model:
                pe[pos][i + 1] = math.cos(angle)
    return pe


def add_positional(embeddings):
    """將 PE 加到 Embedding 向量上"""
    pe = positional_encoding(len(embeddings), len(embeddings[0]))
    return [add_vec(e, p) for e, p in zip(embeddings, pe)]


# 示範
pe = positional_encoding(SEQ, D)
print(f"Positional Encoding 形狀: {len(pe)} x {len(pe[0])}")
print(f"位置 0 的前 6 維: {[f'{x:.4f}' for x in pe[0][:6]]}")
print(f"位置 1 的前 6 維: {[f'{x:.4f}' for x in pe[1][:6]]}")

Positional Encoding 形狀: 16 x 32
位置 0 的前 6 維: ['0.0000', '1.0000', '0.0000', '1.0000', '0.0000', '1.0000']
位置 1 的前 6 維: ['0.8415', '0.5403', '0.5332', '0.8460', '0.3110', '0.9504']


## 5. Self-Attention 層

核心公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

每個位置都能「關注」序列中的所有其他位置。

In [5]:
class SelfAttention:
    """
    單頭 Self-Attention
    參數：Wq, Wk, Wv, Wo  各為 [D x D] 矩陣
    """

    def __init__(self, d_model):
        scale = 0.02
        self.Wq = randn(d_model, d_model, scale)
        self.Wk = randn(d_model, d_model, scale)
        self.Wv = randn(d_model, d_model, scale)
        self.Wo = randn(d_model, d_model, scale)
        self.d_model = d_model

    def forward(self, X, mask=True):
        """
        X: list of T 個 D 維向量
        mask=True 表示 causal masking（只看過去）
        """
        T = len(X)
        dk = math.sqrt(self.d_model)

        # X 轉為矩陣 [T x D]
        Q = matmul(X, self.Wq)  # [T x D]
        K = matmul(X, self.Wk)  # [T x D]
        V = matmul(X, self.Wv)  # [T x D]

        Kt = transpose(K)  # [D x T]

        # 注意力分數 [T x T]
        scores = matmul(Q, Kt)  # [T x T]

        # Causal mask：未來位置設為 -inf
        for i in range(T):
            for j in range(T):
                scores[i][j] /= dk
                if mask and j > i:
                    scores[i][j] = -1e9

        # Softmax over 每一行
        attn = [softmax(scores[i]) for i in range(T)]  # [T x T]

        # 加權求和 V
        out = matmul(attn, V)  # [T x D]

        # 輸出投影
        out = matmul(out, self.Wo)  # [T x D]
        return out

    def params(self):
        return [self.Wq, self.Wk, self.Wv, self.Wo]


# 示範
attn = SelfAttention(D)
dummy_input = [list(pe[i]) for i in range(5)]
out = attn.forward(dummy_input)
print(f"Self-Attention 輸出形狀: {len(out)} x {len(out[0])}")
print(f"第一個輸出向量（前 5 維）: {[f'{x:.4f}' for x in out[0][:5]]}")

Self-Attention 輸出形狀: 5 x 32
第一個輸出向量（前 5 維）: ['0.0018', '0.0161', '0.0087', '-0.0121', '-0.0016']


## 6. Feed Forward Network

兩層線性變換 + ReLU：
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

In [6]:
class FeedForward:
    """
    Position-wise Feed Forward Network
    每個位置獨立通過兩層 MLP
    """

    def __init__(self, d_model, d_ff):
        self.W1 = randn(d_model, d_ff, 0.02)
        self.b1 = [0.0] * d_ff
        self.W2 = randn(d_ff, d_model, 0.02)
        self.b2 = [0.0] * d_model

    def forward(self, X):
        """X: list of T 個 D 維向量 → 相同形狀輸出"""
        result = []
        for x in X:
            # 第一層：Linear + ReLU
            h = [
                sum(x[k] * self.W1[k][j] for k in range(len(x))) + self.b1[j]
                for j in range(len(self.b1))
            ]
            h = relu(h)
            # 第二層：Linear
            out = [
                sum(h[k] * self.W2[k][j] for k in range(len(h))) + self.b2[j]
                for j in range(len(self.b2))
            ]
            result.append(out)
        return result

    def params(self):
        return [self.W1, self.W2]


print("FeedForward 定義完成 ✓")

FeedForward 定義完成 ✓


## 7. Transformer Block

組合 Self-Attention + Feed Forward + 殘差連接 + Layer Norm。

In [7]:
class TransformerBlock:
    """
    單個 Transformer 層：
      x = x + SelfAttention(LayerNorm(x))
      x = x + FFN(LayerNorm(x))
    """

    def __init__(self, d_model, d_ff):
        self.attn = SelfAttention(d_model)
        self.ff = FeedForward(d_model, d_ff)

    def forward(self, X):
        # Self-Attention 子層 + 殘差
        normed = [layer_norm(x) for x in X]
        attn_out = self.attn.forward(normed)
        X = [add_vec(x, a) for x, a in zip(X, attn_out)]

        # Feed Forward 子層 + 殘差
        normed = [layer_norm(x) for x in X]
        ff_out = self.ff.forward(normed)
        X = [add_vec(x, f) for x, f in zip(X, ff_out)]
        return X

    def params(self):
        return self.attn.params() + self.ff.params()


print("TransformerBlock 定義完成 ✓")

TransformerBlock 定義完成 ✓


## 8. 完整語言模型

整合所有組件，加上輸出投影層（Linear → 詞彙表大小）。

In [8]:
class MiniLLM:
    """
    最小可運行 Transformer 語言模型
    架構：Embedding → +PE → TransformerBlock × N → Linear → Softmax
    """

    def __init__(self, vocab_size, d_model, d_ff, n_layers=1):
        self.emb = Embedding(vocab_size, d_model)
        self.blocks = [TransformerBlock(d_model, d_ff) for _ in range(n_layers)]
        self.Wout = randn(d_model, vocab_size, 0.02)  # 輸出投影
        self.d_model = d_model
        self.vocab_size = vocab_size

    def forward(self, ids):
        """ids: list of int → 每個位置的 logits list"""
        # Embedding
        X = self.emb.forward(ids)  # [T x D]
        # 加入 Positional Encoding
        X = add_positional(X)
        # Transformer blocks
        for block in self.blocks:
            X = block.forward(X)
        # Layer Norm 最後一層
        X = [layer_norm(x) for x in X]
        # 輸出投影：每個位置 → vocab 大小的 logits
        logits = []
        for x in X:
            row = [
                sum(x[k] * self.Wout[k][j] for k in range(self.d_model))
                for j in range(self.vocab_size)
            ]
            logits.append(row)
        return logits  # [T x vocab_size]

    def all_params(self):
        """取得所有可訓練參數（矩陣列表）"""
        p = self.emb.params() + [self.Wout]
        for b in self.blocks:
            p += b.params()
        return p


# 建立模型
random.seed(42)
model = MiniLLM(vocab_size=V, d_model=D, d_ff=D_FF, n_layers=1)

# 快速前向傳播測試
test_text = "hello"
test_ids = [char2id.get(c, 0) for c in test_text]
logits = model.forward(test_ids)
print(f"輸入: '{test_text}'  ({len(test_ids)} tokens)")
print(f"輸出 logits 形狀: {len(logits)} x {len(logits[0])}")
probs = softmax(logits[-1])
top_id = probs.index(max(probs))
print(f"預測下一個字元: '{id2char[top_id]}'（未訓練，隨機）")

輸入: 'hello'  (5 tokens)
輸出 logits 形狀: 5 x 32
預測下一個字元: 't'（未訓練，隨機）


## 9. 損失函數與訓練

使用 **Cross-Entropy Loss** 訓練模型預測下一個 Token，並用數值微分（有限差分法）計算梯度。

In [9]:
def cross_entropy_loss(logits_seq, target_ids):
    """
    計算序列的平均 Cross-Entropy Loss
    logits_seq: [T x V]，target_ids: [T]（預測下一個字）
    使用 input[:-1] 預測 target[1:]（標準 LM 訓練方式）
    """
    total_loss = 0.0
    count = 0
    for t in range(len(target_ids)):
        probs = softmax(logits_seq[t])
        target = target_ids[t]
        total_loss -= math.log(probs[target] + 1e-9)
        count += 1
    return total_loss / count if count > 0 else 0.0


def compute_loss(model, text):
    """對一段文字計算語言模型損失"""
    ids = [char2id.get(c, 0) for c in text]
    if len(ids) < 2:
        return 0.0
    input_ids = ids[:-1]  # 輸入：去掉最後一個
    target_ids = ids[1:]  # 目標：去掉第一個（預測下一個）
    logits = model.forward(input_ids)
    return cross_entropy_loss(logits, target_ids)


def numerical_gradient(model, text, eps=1e-3):
    """
    數值微分（有限差分法）計算所有參數的梯度
    grad ≈ (f(x+ε) - f(x-ε)) / (2ε)
    注意：這很慢，僅用於教學示範
    """
    grads = []
    for param_matrix in model.all_params():
        grad_matrix = zeros(len(param_matrix), len(param_matrix[0]))
        for i in range(len(param_matrix)):
            for j in range(len(param_matrix[0])):
                orig = param_matrix[i][j]
                param_matrix[i][j] = orig + eps
                loss_plus = compute_loss(model, text)
                param_matrix[i][j] = orig - eps
                loss_minus = compute_loss(model, text)
                param_matrix[i][j] = orig
                grad_matrix[i][j] = (loss_plus - loss_minus) / (2 * eps)
        grads.append(grad_matrix)
    return grads


def update_params(model, grads, lr):
    """梯度下降更新所有參數"""
    for param_matrix, grad_matrix in zip(model.all_params(), grads):
        for i in range(len(param_matrix)):
            for j in range(len(param_matrix[0])):
                param_matrix[i][j] -= lr * grad_matrix[i][j]


# 測試損失計算
loss = compute_loss(model, "hello")
print(f"未訓練時的損失: {loss:.4f}")
print(f"（隨機猜測的理論損失: {math.log(V):.4f}）")

未訓練時的損失: 3.4343
（隨機猜測的理論損失: 3.4657）


## 10. 訓練迴圈

⚠️ **注意**：數值微分非常慢（O(參數數量) 次前向傳播），這裡用少量迭代示範訓練過程。

實際生產環境會用反向傳播（autograd）來加速。

In [10]:
# 訓練資料：讓模型記住一小段文字
train_text = "hello world"

print(f"訓練文字: '{train_text}'")
print(f"字元數: {len(train_text)}")
print("\n開始訓練（數值微分，每步需要一些時間）...\n")

# 重新初始化模型（較小以加速示範）
random.seed(42)
D_SMALL = 16
DFF_SMALL = 32
model_small = MiniLLM(vocab_size=V, d_model=D_SMALL, d_ff=DFF_SMALL, n_layers=1)

N_STEPS = 10  # 示範用 10 步（可增加以獲得更好結果）
losses = []

for step in range(N_STEPS):
    loss = compute_loss(model_small, train_text)
    losses.append(loss)

    # 僅對 Embedding 矩陣計算梯度（加速示範）
    eps = 1e-3
    param = model_small.emb.W
    for i in range(len(param)):
        for j in range(len(param[0])):
            orig = param[i][j]
            param[i][j] = orig + eps
            lp = compute_loss(model_small, train_text)
            param[i][j] = orig - eps
            lm = compute_loss(model_small, train_text)
            param[i][j] = orig
            grad = (lp - lm) / (2 * eps)
            param[i][j] -= LR * grad

    print(f"Step {step+1:3d}/{N_STEPS}  Loss: {loss:.4f}")

print(f"\n訓練完成！損失從 {losses[0]:.4f} 降至 {losses[-1]:.4f}")

訓練文字: 'hello world'
字元數: 11

開始訓練（數值微分，每步需要一些時間）...

Step   1/10  Loss: 3.4398
Step   2/10  Loss: 3.4397
Step   3/10  Loss: 3.4397
Step   4/10  Loss: 3.4397
Step   5/10  Loss: 3.4397
Step   6/10  Loss: 3.4397
Step   7/10  Loss: 3.4397
Step   8/10  Loss: 3.4397
Step   9/10  Loss: 3.4397
Step  10/10  Loss: 3.4397

訓練完成！損失從 3.4398 降至 3.4397


## 11. 文字生成

訓練完成後，用模型「補全」文字（Autoregressive 生成）。

In [11]:
def generate(model, prompt, max_new_tokens=20, temperature=1.0):
    """
    自回歸生成文字
    temperature > 1 → 更隨機；temperature < 1 → 更確定
    """
    ids = [char2id.get(c, 0) for c in prompt]
    generated = list(prompt)

    for _ in range(max_new_tokens):
        # 只取最近 SEQ 個 token（滑動視窗）
        context = ids[-SEQ:]
        logits = model.forward(context)
        # 取最後一個位置的 logits
        last_logits = logits[-1]
        # 套用 temperature
        scaled = [l / temperature for l in last_logits]
        probs = softmax(scaled)
        # 依機率採樣
        r = random.random()
        cumsum = 0.0
        next_id = 0
        for idx, p in enumerate(probs):
            cumsum += p
            if r < cumsum:
                next_id = idx
                break
        ids.append(next_id)
        generated.append(id2char[next_id])

    return "".join(generated)


# 生成示範
random.seed(0)
prompt = "h"
print(f"Prompt: '{prompt}'")

for temp, label in [(0.5, "低溫（保守）"), (1.0, "標準"), (2.0, "高溫（創意）")]:
    result = generate(model_small, prompt, max_new_tokens=15, temperature=temp)
    print(f"  temperature={temp} ({label}): '{result}'")

print("\n（模型訓練步數極少，生成結果仍偏隨機；增加訓練步數可改善）")

Prompt: 'h'
  temperature=0.5 (低溫（保守）): 'h.yniqmzjos,pixt'
  temperature=1.0 (標準): 'hi,
z,jx,vodnt,?'
  temperature=2.0 (高溫（創意）): 'hp.izraxm vap.hk'

（模型訓練步數極少，生成結果仍偏隨機；增加訓練步數可改善）


## 12. 整合：端對端示範

In [12]:
# 查看損失下降趨勢
print("=" * 40)
print("損失下降紀錄")
print("=" * 40)
for i, l in enumerate(losses):
    bar = "█" * int(l * 5)
    print(f"Step {i+1:2d}: {l:.4f}  {bar}")

print("\n" + "=" * 40)
print("模型參數量統計")
print("=" * 40)
total = 0
for pm in model_small.all_params():
    count = len(pm) * len(pm[0])
    total += count
print(f"總參數量: {total:,}")
print(f"（GPT-2 有 1.17 億參數；GPT-4 估計超過 1 兆）")

損失下降紀錄
Step  1: 3.4398  █████████████████
Step  2: 3.4397  █████████████████
Step  3: 3.4397  █████████████████
Step  4: 3.4397  █████████████████
Step  5: 3.4397  █████████████████
Step  6: 3.4397  █████████████████
Step  7: 3.4397  █████████████████
Step  8: 3.4397  █████████████████
Step  9: 3.4397  █████████████████
Step 10: 3.4397  █████████████████

模型參數量統計
總參數量: 3,072
（GPT-2 有 1.17 億參數；GPT-4 估計超過 1 兆）


## 延伸學習

這個迷你 LLM 展示了核心概念，但真實 LLM 還有許多改進：

| 功能 | 本示範 | 真實 LLM |
|------|--------|----------|
| 梯度計算 | 數值微分（慢）| 反向傳播（autograd）|
| 矩陣運算 | 純 Python | NumPy / CUDA GPU |
| 詞彙表 | 字元級（32）| BPE（50,000+）|
| 參數量 | ~幾千 | 數十億 |
| 注意力 | 單頭 | 多頭（Multi-Head）|
| 訓練資料 | 幾個詞 | 萬億 Token |

**推薦下一步：**
- 🔗 [Andrej Karpathy 的 nanoGPT](https://github.com/karpathy/nanoGPT)（真正的小型 GPT）
- 📖 Attention Is All You Need（原始 Transformer 論文）
- 🛠️ 用 PyTorch 實作自動微分版本